In [ ]:
import boto3
import json
from typing import Optional

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv(override=True) 

In [ ]:
AWS_REGION = "ap-southeast-2"

In [ ]:
MODELS = {
    # Available directly in Sydney:
    "claude_35_sonnet_v2": "anthropic.claude-3-5-sonnet-20241022-v2:0",  # Best for POC - recommended
    "claude_3_haiku": "anthropic.claude-3-haiku-20240307-v1:0",          # Fastest, cheapest
    "claude_3_sonnet": "anthropic.claude-3-sonnet-20240229-v1:0",        # Older sonnet
}

In [ ]:
api_key = os.environ.get('AWS_BEARER_TOKEN_BEDROCK')

In [ ]:
api_key

In [ ]:
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-southeast-2"
)

In [ ]:
DEFAULT_MODEL = MODELS["claude_3_haiku"]

In [ ]:
def check_model_access(region: str = AWS_REGION):
    """List available foundation models to verify access."""
    bedrock = boto3.client("bedrock", region_name=region)
    
    response = bedrock.list_foundation_models(byProvider="Anthropic")
    
    print("Available Anthropic models in your account:\n")
    for model in response["modelSummaries"]:
        print(f"  • {model['modelId']}")
        print(f"    Status: {model.get('modelLifecycle', {}).get('status', 'N/A')}")
        print()
    
    return response

In [ ]:
class BedrockClient:
    """Simple wrapper for Bedrock inference - suitable for POC experimentation."""
    
    def __init__(self, region: str = AWS_REGION, model_id: str = DEFAULT_MODEL):
        # Note: bedrock-RUNTIME for inference, not just "bedrock"
        self.client = boto3.client("bedrock-runtime", region_name=region)
        self.model_id = model_id
        
    def invoke(
        self,
        prompt: str,
        system: Optional[str] = None,
        max_tokens: int = 1024,
        temperature: float = 0.0,  # Use 0 for deterministic categorisation
    ) -> dict:
        """
        Invoke Claude model with the Messages API format.
        
        ⚠️ KEY PARAMETERS:
        - temperature: 0.0 for consistent categorisation, higher for creativity
        - max_tokens: Keep low for categorisation (saves cost), increase for explanations
        """
        
        # Build request body - Claude uses the Messages API format
        body = {
            "anthropic_version": "bedrock-2023-05-31",  # Required!
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
        
        # Add system prompt if provided
        if system:
            body["system"] = system
        
        response = self.client.invoke_model(
            modelId=self.model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(body)
        )
        
        # Parse response
        result = json.loads(response["body"].read())
        
        # Return structured result with usage stats (important for cost tracking!)
        return {
            "content": result["content"][0]["text"],
            "usage": result.get("usage", {}),  # Track tokens for cost analysis
            "model": self.model_id,
            "stop_reason": result.get("stop_reason")
        }


In [ ]:
bedrock = BedrockClient()

In [ ]:
response = bedrock.invoke("Say 'Bedrock connection successful!' and nothing else.")
print(f"Response: {response['content']}")
print(f"Tokens used: {response['usage']}")